# Set up a regional MOM6 run within CESM framework (with CRR grid generation)

There are three main sections: 
1. First we create the MOM6 experiment using the [regional-mom6](https://github.com/COSIMA/regional-mom6/) package. This puts everything we need into two repositories, referred to as the `mom_run_dir` and `mom_input_dir`. These contain all of the configuration files (`MOM_input` etc.) and netcdf input files (`hgrid.nc`, OBC segments etc.) respectively. An `experiment` object is also created within the notebook, containing all the information we need about the regional mom6 setup to pass onto CESM.

2. A new MOM6 CESM case is created in the usual way by cloning the CESM repo and running the `create_case` command. 

3. We modify the new CESM case to work with the regional mom6 configuration we prepared earlier. We pass the path to this CESM run and the experiment object to the `setup_cesm` function, which carries out all the required modifications 

This is very much a work in progress! The goal in the short term was to get something working and make it nice later

# SECTION 1: Setup up you MOM6 regional experiment

This follows the normal workflow, copied from the `reanalysis_forced.ipynb` demo of [regional-mom6](https://github.com/COSIMA/regional-mom6/) but modified for a domain around Hawaii. See the documentation of the package for details and documentation

In [1]:
from pathlib import Path
import sys

In [2]:


sys.path.insert(0, '/glade/u/home/manishrv/documents/nwa12_0.1/regional_mom_workflows/crr')
import crocodileregionalruckus as crr
from crocodileregionalruckus.rm6 import regional_mom6 as rmom6

In [3]:
test_num = 1
test_name = f"{test_num}.test.tides"

In [4]:
import json 
with open('/glade/u/home/manishrv/documents/nwa12_0.1/configs/nwa_config.json', 'r') as file:
    data = json.load(file)

# Step 2: Change the file path in the JSON (assuming the path is in a specific key)
data['mom_input_dir'] = f'/glade/u/home/manishrv/documents/nwa12_0.1/mom_input/{test_name}'
data['mom_run_dir'] = f'/glade/u/home/manishrv/documents/nwa12_0.1/mom_run/{test_name}'
data['minimum_depth'] = 9.95
data['tides'] = []
# Step 3: Save the modified JSON back to the file
with open(f'/glade/u/home/manishrv/documents/nwa12_0.1/configs/{test_name}_nwa_config.json', 'w') as file:
    json.dump(data, file, indent=4)

In [5]:
expt =crr.driver.crr_driver.create_experiment_from_config(f"/glade/u/home/manishrv/documents/nwa12_0.1/configs/{test_name}_nwa_config.json")

2024-10-28 11:26:50,938 - crocodileregionalruckus.driver - INFO - Reading from config file....
2024-10-28 11:26:50,941 - crocodileregionalruckus.driver - INFO - Creating Input and Run Dirs if not already....
2024-10-28 11:26:50,973 - crocodileregionalruckus.driver - INFO - Checking for hgrid and vgrid....
2024-10-28 11:26:50,979 - crocodileregionalruckus.driver - INFO - Found
2024-10-28 11:26:50,982 - crocodileregionalruckus.driver - WARNING - There was a hgrid.nc already in the directory to replace. We are removing it! To disable this aggresive move, set rearrange_files_to_expt_format=False
2024-10-28 11:26:50,988 - crocodileregionalruckus.driver - INFO - Copying hgrid.nc to /glade/u/home/manishrv/documents/nwa12_0.1/mom_input/1.test.tides/hgrid.nc
2024-10-28 11:26:51,172 - crocodileregionalruckus.driver - INFO - Found
2024-10-28 11:26:51,176 - crocodileregionalruckus.driver - WARNING - There was a vcoord.nc already in the directory to replace. We are removing it! To disable this aggr

In [6]:
import os
ocean_varnames = {"time": "time",
                  "yh": "latitude",
                  "xh": "longitude",
                  "zl": "depth",
                  "eta": "zos",
                  "u": "uo",
                  "v": "vo",
                  "tracers": {"salt": "so", "temp": "thetao"}
                  }

glorys_path = os.path.join("/","glade","derecho","scratch","manishrv","inputs_rm6_nwa","glorys" )
expt.setup_ocean_state_boundaries(
         Path(glorys_path),
         ocean_varnames,
         arakawa_grid = "A"
         )

NOTE: the 'setup_run_directories' method does understand the less than four boundaries but be careful. Please check the MOM_input/override file carefully to reflect the number of boundaries you have, and their orientations. You should be able to find the relevant section in the MOM_input/override file by searching for 'segment_'. Ensure that the segment names match those in your inputdir/forcing folder
[########################################] | 100% Completed | 104.37 ms
Done.
[########################################] | 100% Completed | 141.88 ms
Done.
[########################################] | 100% Completed | 113.88 ms
Done.


In [9]:
sys.modules["regional_mom6"] = rmom6
expt.setup_run_directory(surface_forcing = "jra", with_tides=True)

Could not find premade run directories at  /glade/u/home/manishrv/documents/nwa12_0.1/regional_mom_workflows/crr/crocodileregionalruckus/rm6/regional_mom6/demos/premade_run_directories
Perhaps the package was imported directly rather than installed with conda. Checking if this is the case... 
Found run files. Continuing...
At least one of these is requiret to set up the experiment if you're running MOM6 standalone with the FMS coupler. 
If you're running within CESM, ignore this message.
Deleting indexed OBC keys from MOM_input_dict in case we have a different number of segments
Changed OBC_SEGMENT_001_DATA from {'value': '"U=file:forcing_obc_segment_001.nc(u),V=file:forcing_obc_segment_001.nc(v),SSH=file:forcing_obc_segment_001.nc(eta),TEMP=file:forcing_obc_segment_001.nc(temp),SALT=file:forcing_obc_segment_001.nc(salt)"', 'override': True, 'comment': 'None\n'} to {'value': '"U=file:forcing_obc_segment_001.nc(u),V=file:forcing_obc_segment_001.nc(v),SSH=file:forcing_obc_segment_001.nc(

# SECTION 2: Create a blank CESM run

So far I've used Alper's [GUI](https://github.com/ESMCI/visualCaseGen?tab=readme-ov-file) branch of CESM. Clone respective branch of CESM and then run the generate case command. Below is the command I used to generate a global, MOM6 only run forced with JRA data atmosphere

`/glade/u/home/abarnes/cesm-runs/visualCaseGen/cesm2_3_beta17_gui/cime/scripts/create_newcase --compset 1850_DATM%JRA_SLND_SICE_MOM6_SROF_SGLC_SWAV --res TL319_t232 --case /glade/u/home/manishrv/cases/{test_name} --machine derecho --run-unsupported --project p93300012 --non-local`

In [11]:
# Save the path where the new CESM config lives
print(f"/glade/u/home/abarnes/cesm-runs/visualCaseGen/cesm2_3_beta17_gui/cime/scripts/create_newcase --compset 1850_DATM%JRA_SLND_SICE_MOM6_SROF_SGLC_SWAV --res TL319_t232 --case /glade/u/home/manishrv/cases/{test_name} --machine derecho --run-unsupported --project p93300012 --non-local")
CESM_path = Path(f"/glade/u/home/manishrv/cases/{test_name}")

/glade/u/home/abarnes/cesm-runs/visualCaseGen/cesm2_3_beta17_gui/cime/scripts/create_newcase --compset 1850_DATM%JRA_SLND_SICE_MOM6_SROF_SGLC_SWAV --res TL319_t232 --case /glade/u/home/manishrv/cases/2.test.tides --machine derecho --run-unsupported --project p93300012 --non-local


# SECTION 3: Modify the CESM run to make it regional

In [12]:
## Regional CaseGen is part of crr

reg_cas_obj = crr.regional_casegen.cesm_tools.RegionalCaseGen()


reg_cas_obj.setup_cesm(expt, CESM_path, project = "p93300612")
## OR you can explicitly state what variables are needed
# reg_cas_obj.setup_cesm_explicit(hgrid = expt.hgrid, CESMPath = CESM_path,project = "p93300612", mom_input_dir=expt.mom_input_dir, mom_run_dir=expt.mom_run_dir,date_range = expt.date_range)

2024-10-25 14:01:25,315 - crocodileregionalruckus.regional_casegen.cesm_tools - INFO - Copying input.nml, diag_table, MOM_input_and MOM_override to /glade/u/home/manishrv/cases/2.test.tides/SourceMods/src.mom
2024-10-25 14:01:25,593 - crocodileregionalruckus.regional_casegen.cesm_tools - INFO - Adding NIGLOBAL = 740, NJGLOBAL = 780, and INPUTDIR = /glade/u/home/manishrv/documents/nwa12_0.1/mom_input/2.test.tides to MOM_override
2024-10-25 14:01:25,594 - crocodileregionalruckus.regional_casegen.cesm_tools - INFO - Add MOM_override to parameter_filename in input.nml
2024-10-25 14:01:25,594 - crocodileregionalruckus.regional_casegen.cesm_tools - INFO - Removing reference to MOM_layout in input.nml and add MOM_override to parameter_filename in input.nml
2024-10-25 14:01:25,595 - crocodileregionalruckus.regional_casegen.cesm_tools - INFO - 
2024-10-25 14:01:25,595 - crocodileregionalruckus.regional_casegen.cesm_tools - INFO - Modifying line: parameter_filename = 'MOM_input', 'MOM_layout', '

Now hopefully doing the usual 

`./case.setup && ./case.build && ./case.submit` 

should at least run. Of course from here you'll have a lot of other things to fiddle around with to make it run *well!* 